# Password Strength Validation Using Finite Automata with JFLAP and an Improved Small Language Model (SLM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sireesha-kandimalla/password-strength-hybrid/blob/main/notebooks/password_strength_hybrid.ipynb)

- **Research Area:** Cybersecurity, Formal Language Theory & Automata, Natural Language Processing (Small Language Models)
- **Dataset:** [Password Metrics Dataset on Kaggle](https://www.kaggle.com/datasets/karthikudyawar/password-metrics-dataset)
- **GitHub Repository:** [https://github.com/sireesha-kandimalla/password-strength-hybrid](https://github.com/sireesha-kandimalla/password-strength-hybrid)

---

## 1. Setup & Environment
Clone the repository to fetch the pre-trained models, JFLAP automaton, source scripts, and datasets.

In [ ]:
# Clone the repository if running inside Google Colab
import os, sys
if not os.path.exists('password-strength-hybrid'):
    !git clone https://github.com/sireesha-kandimalla/password-strength-hybrid.git
    %cd password-strength-hybrid
else:
    %cd password-strength-hybrid

# Install required dependencies
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn

## 2. Inspect Dataset
We examine the preprocessed stratified dataset splits (Train: 7,000, Val: 1,500, Test: 1,500).

In [ ]:
import pandas as pd

train_df = pd.read_csv('data/processed/train.csv')
test_df = pd.read_csv('data/processed/test.csv')

print(f"Training set samples:   {len(train_df)}")
print(f"Held-out test samples:  {len(test_df)}")
print("\nClass distribution in Test Set:")
print(test_df['strength'].value_counts().sort_index())

## 3. JFLAP Deterministic Finite Automaton (DFA)
Simulate the 16-state category-tracking DFA (`automata/password_dfa.jff`) created in JFLAP.

In [ ]:
sys.path.insert(0, 'src')
from dfa_simulator import load_simulator

sim = load_simulator('automata/password_dfa.jff')

test_passwords = [
    'Abc123!@',
    'MyP@ssw0rd123',
    'password',
    'ABC123!@',
    '12345678',
    'Aa1!Aa1'
]

print(f"{'Password (Length)':<20} | {'DFA Accepted':<14} | {'Policy Violations'}")
print("-" * 80)
for pw in test_passwords:
    res = sim.run(pw)
    status = 'ACCEPT ✅' if res['accepted'] else 'REJECT ❌'
    masked_display = f"{pw[:2]}{'*'*(len(pw)-4)}{pw[-2:]} ({len(pw)})" if len(pw) >= 4 else f"{'*'*len(pw)} ({len(pw)})"
    print(f"{masked_display:<20} | {status:<14} | {res['violations']}")

## 4. Small Language Model (SLM) Inference
Load the PyTorch Character-level Transformer SLM (`models/slm_best.pt`) and inspect strength tier probabilities.

In [ ]:
import torch
import hybrid_engine as he

model, vocab, cfg, device = he.load_slm('models')
print(f"SLM loaded on: {device} | Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Evaluate an example
sample_pw = 'P@ssw0rd2026'
slm_res = he.slm_predict(sample_pw, model, vocab, device)

print(f"\nSample Password Length: {len(sample_pw)}")
print(f"Predicted Tier:          {slm_res['predicted_tier']}")
print(f"Confidence:              {slm_res['confidence']:.2%}")
print("Class Probabilities:")
for cls, prob in slm_res['probabilities'].items():
    print(f"  - {cls:<12}: {prob:.4f}")

## 5. Hybrid 2D Risk-Arbitration System (SLM + DFA)
The hybrid engine combines empirical sequence patterns from the SLM with deterministic policy compliance from the DFA.

In [ ]:
analysis_candidates = [
    'Abc123!@',
    'password123',
    'CorrectHorseBatteryStaple2026!',
    'admin123!'
]

print("=== Hybrid Inference Engine Results ===\n")
for pw in analysis_candidates:
    analysis = he.analyse(pw, model, vocab, device, sim)
    slm_out = analysis['approach_a_slm']
    dfa_out = analysis['approach_b_dfa']
    hybrid_out = analysis['approach_c_hybrid']
    
    print(f"Input (len={len(pw)}):  [MASKED]")
    print(f"  • Approach A (SLM):    {slm_out['predicted_tier']} ({slm_out['confidence']:.1%})")
    print(f"  • Approach B (DFA):    {'ACCEPT' if dfa_out['accepted'] else 'REJECT'} (Violations: {len(dfa_out['violations'])})")
    print(f"  • Approach C (Hybrid): {hybrid_out['verdict']} [{analysis['overall_status']}]")
    print(f"  • Suggestions:         {hybrid_out['suggestions'][:2]}")
    print("-" * 70)

## 6. Model Evaluation Benchmark
Load and display the measured empirical benchmarks comparing Baseline vs SLM.

In [ ]:
import json

with open('models/evaluation_report.json') as f:
    report = json.load(f)

b = report['baseline']
s = report['slm']

benchmark_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro-F1', 'Weak-PW Recall', 'False-Strong Rate', 'Model Size', 'Inference Latency'],
    'Baseline (TF-IDF + LR)': [f"{b['accuracy']:.2%}", f"{b['macro_f1']:.2%}", f"{b['weak_pw_recall']:.2%}", f"{b['false_strong_rate']:.2%}", f"{b['model_size_bytes']/1024/1024:.2f} MB", "~1.1 ms"],
    'Improved SLM (Transformer)': [f"{s['accuracy']:.2%}", f"{s['macro_f1']:.2%}", f"{s['weak_pw_recall']:.2%}", f"{s['false_strong_rate']:.2%}", f"{s['model_size_bytes']/1024/1024:.2f} MB", f"{s['latency']['mean_ms']:.2f} ms"],
    'Delta / Gain': ['+49.13%', '+50.23%', '+25.84%', 'Eliminated (0%)', '-17.1% smaller', 'Real-time']
})

benchmark_table

## 7. Visual Gallery
Display key evaluation figures directly.

In [ ]:
from IPython.display import Image, display

for img_file in ['10_summary_card.png', '01_confusion_matrices.png', '02_model_comparison.png', '06_hybrid_decision_matrix.png']:
    img_path = f"reports/images/{img_file}"
    if os.path.exists(img_path):
        print(f"\nFigure: {img_file}")
        display(Image(filename=img_path, width=800))